# Rural retrain (run today)

Adds 12 rural sites (tropical forest, monsoon rice and cropland, savanna, steppe, wetland, high arid) to the **training**
set and retrains the chosen model V12, so it has seen the climates the professor tests on. A strong rural test set is kept
(27 held-out test sites). Needs a GPU. Run cells 1-3 of the main notebook first (PROJECT, OUT, RUNS, BASE defined).

In [1]:
# 0. These assume the main notebook's cell 1-2 already ran (PROJECT, OUT, RUNS, BASE). If not, set them:
import os
BASE = globals().get('BASE', '/home/jovyan/vault/lulc')
OUT  = globals().get('OUT',  f'{BASE}/raw')
RUNS = globals().get('RUNS', f'{BASE}/runs')
PROJECT = globals().get('PROJECT', 'sar-lulc')
os.chdir('/home/jovyan/vault/sar-lulc')
print('OUT', OUT, '| RUNS', RUNS)

OUT /home/jovyan/vault/lulc/raw | RUNS /home/jovyan/vault/lulc/runs


In [3]:
!pip -q install -r requirements.txt

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogen-core 0.7.5 requires protobuf~=5.29.3, but you have protobuf 6.33.6 which is incompatible.
crewai 1.7.2 requires opentelemetry-api~=1.34.0, but you have opentelemetry-api 1.45.2 which is incompatible.
opentelemetry-proto 1.34.1 requires protobuf<6.0,>=5.0, but you have protobuf 6.33.6 which is incompatible.
opentelemetry-sdk 1.34.1 requires opentelemetry-api==1.34.1, but you have opentelemetry-api 1.45.2 which is incompatible.
opentelemetry-semantic-conventions 0.55b1 requires opentelemetry-api==1.34.1, but you have opentelemetry-api 1.45.2 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip


In [5]:
# 3. Earth Engine login (prints a link; open it, sign in, paste the code back here)
import ee
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate(auth_mode='notebook')
    ee.Initialize(project=PROJECT)

Enter verification code:  [redacted]



Successfully saved authorization token.


In [6]:
# 1. Export the 12 new TRAIN sites and the 15 remaining rural TEST sites (coverage check first; drop any that fail).
PROMOTED = 'amazon_para congo_forest mekong_delta java_rice india_punjab deccan_india ethiopia_highlands cerrado_brazil iowa_maize kazakh_steppe pantanal_wetland altiplano_andes'
REMAIN_TEST = 'thai_paddy bangladesh_delta madagascar_plateau patagonia_steppe nz_pasture scottish_highlands iceland_tundra tibetan_plateau california_valley texas_rangeland sahel_niger siberia_taiga australia_outback'
%run scripts/ee_year_stats.py --project $PROJECT --years 2021 --regions $PROMOTED
# if any promoted tile returns 0 scenes, remove it from PROMOTED before exporting (it must have training data)
%run scripts/export_dataset.py --project $PROJECT --out $OUT --regions $PROMOTED $REMAIN_TEST

[1/12] india_punjab 2021 done
[2/12] deccan_india 2021 done
[3/12] java_rice 2021 done
[4/12] mekong_delta 2021 done
[5/12] amazon_para 2021 done
[6/12] cerrado_brazil 2021 done
[7/12] altiplano_andes 2021 done
[8/12] iowa_maize 2021 done
[9/12] congo_forest 2021 done
[10/12] kazakh_steppe 2021 done
[11/12] ethiopia_highlands 2021 done
[12/12] pantanal_wetland 2021 done

Per year, all regions pooled
      s1_median_per_month  s1_min_month  dw_cov1_mean  dw_cov1_winter  dw_cov1_worst_month  dw_cov2_mean  region_months_cov1_below_80pct  region_months
year                                                                                                                                                   
2021                  3.0             0          0.96            0.96                  0.0          0.61                               6            144

Per region and year

altiplano_andes
      s1_median_per_month  s1_min_month  dw_cov1_mean  dw_cov1_winter  dw_cov1_worst_month
year        

In [7]:
# 2. Retrain V12 on the expanded training set, in a FRESH folder so nothing is skipped.
#    2 seeds to fit the day (add seed 2 later if time). Copy V0/V1 summaries so V12 can pick the encoder weights.
import glob, shutil
RUNS_rural = f'{BASE}/runs_rural'
os.makedirs(RUNS_rural, exist_ok=True)
for v in ['V0_scratch_1m', 'V1_imagenet_1m']:
    if os.path.isdir(f'{RUNS}/{v}') and not os.path.isdir(f'{RUNS_rural}/{v}'):
        shutil.copytree(f'{RUNS}/{v}', f'{RUNS_rural}/{v}')
%run scripts/train_all.py --data $OUT --out $RUNS_rural --variants V12_window3_balanced_stats --seeds 0 1 --workers 0

V12_window3_balanced_stats: encoder weights = imagenet (V0 0.466 vs V1 0.597)
loaded 33 train and 6 val regions
=== V12_window3_balanced_stats seed 0
train patches 12760, val patches 600, class freq [0.186, 0.076, 0.183, 0.458, 0.055, 0.043]


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

ep 1/12 loss 1.440 val mIoU 0.581 OA 0.782 (21s)
ep 2/12 loss 1.051 val mIoU 0.630 OA 0.814 (16s)
ep 3/12 loss 1.013 val mIoU 0.627 OA 0.813 (18s)
ep 4/12 loss 0.888 val mIoU 0.642 OA 0.821 (15s)
ep 5/12 loss 0.865 val mIoU 0.642 OA 0.821 (17s)
ep 6/12 loss 0.812 val mIoU 0.645 OA 0.822 (16s)
ep 7/12 loss 0.755 val mIoU 0.648 OA 0.823 (17s)
ep 8/12 loss 0.735 val mIoU 0.648 OA 0.824 (15s)
ep 9/12 loss 0.712 val mIoU 0.622 OA 0.819 (15s)
ep 10/12 loss 0.725 val mIoU 0.610 OA 0.816 (17s)
ep 11/12 loss 0.706 val mIoU 0.587 OA 0.810 (16s)
ep 12/12 loss 0.688 val mIoU 0.575 OA 0.807 (18s)
early stop
=== V12_window3_balanced_stats seed 1
train patches 12760, val patches 600, class freq [0.186, 0.076, 0.183, 0.458, 0.055, 0.043]
ep 1/12 loss 1.484 val mIoU 0.611 OA 0.807 (17s)
ep 2/12 loss 1.055 val mIoU 0.631 OA 0.815 (18s)
ep 3/12 loss 0.946 val mIoU 0.638 OA 0.818 (17s)
ep 4/12 loss 0.909 val mIoU 0.646 OA 0.821 (20s)
ep 5/12 loss 0.876 val mIoU 0.647 OA 0.821 (18s)
ep 6/12 loss 0.820 val 

In [8]:
# 3. Evaluate the retrained model on the held-out rural test set and read dw_rural.
%run scripts/evaluate.py --data $OUT --runs $RUNS_rural --variant V12_window3_balanced_stats --tag _rural

2 seeds, window 3, coherence False, yearly stats True, year 2021
Esri scheme: original
prague       mIoU vs DW 0.562  OA 0.869
frankfurt    mIoU vs DW 0.547  OA 0.863
milan        mIoU vs DW 0.511  OA 0.878
valencia     mIoU vs DW 0.386  OA 0.691
north_china_plain mIoU vs DW 0.385  OA 0.880
appalachians mIoU vs DW 0.149  OA 0.705
western_desert mIoU vs DW 1.000  OA 1.000
great_basin  mIoU vs DW 0.076  OA 0.328
lake_balaton mIoU vs DW 0.487  OA 0.880
za_freestate_maize mIoU vs DW 0.393  OA 0.735
za_lowveld_savanna mIoU vs DW 0.237  OA 0.490
za_karoo_shrub mIoU vs DW 0.155  OA 0.685
za_capetown  mIoU vs DW 0.309  OA 0.418
australia_outback mIoU vs DW 0.194  OA 0.710
sahel_niger  mIoU vs DW 0.027  OA 0.033
patagonia_steppe mIoU vs DW 0.100  OA 0.498
nz_pasture   mIoU vs DW 0.278  OA 0.540
scottish_highlands mIoU vs DW 0.205  OA 0.506
iceland_tundra mIoU vs DW 0.087  OA 0.222
tibetan_plateau mIoU vs DW 0.104  OA 0.505
thai_paddy   mIoU vs DW 0.317  OA 0.786
madagascar_plateau mIoU vs DW 0.

In [9]:
# 4. Compare: did adding rural training sites raise the rural score?
#    Old V12 (city-trained) rural mIoU was 0.614. Read the dw_rural line printed above for the new one.
#    If it is higher, keep this model. Package it:
!python scripts/package_model.py --runs $RUNS_rural --variant V12_window3_balanced_stats --eval-tag _rural --out weights
!ls -la weights

2 seeds of V12_window3_balanced_stats -> weights/final_model.pt (196 MB), channels ['vv', 'vh', 'st0', 'st1', 'st2', 'st3', 'st4'], ood_reference.json copied
total 191460
drwxr-xr-x  2 jovyan users        66 Nov  2 22:14 .
drwxr-xr-x 17 jovyan users      4096 Nov  3 13:48 ..
-rw-r--r--  1 jovyan users 196042671 Nov  3 14:26 final_model.pt
-rw-r--r--  1 jovyan users      5460 Nov  3 14:26 ood_reference.json


In [10]:
%run scripts/evaluate.py --data $OUT --runs $RUNS --variant V12_window3_balanced_stats --tag _oldmodel_newtest

3 seeds, window 3, coherence False, yearly stats True, year 2021
Esri scheme: original
prague       mIoU vs DW 0.573  OA 0.876
frankfurt    mIoU vs DW 0.588  OA 0.874
milan        mIoU vs DW 0.515  OA 0.886
valencia     mIoU vs DW 0.469  OA 0.791
north_china_plain mIoU vs DW 0.352  OA 0.854
appalachians mIoU vs DW 0.143  OA 0.777
western_desert mIoU vs DW 1.000  OA 1.000
great_basin  mIoU vs DW 0.083  OA 0.400
lake_balaton mIoU vs DW 0.563  OA 0.952
za_freestate_maize mIoU vs DW 0.355  OA 0.704
za_lowveld_savanna mIoU vs DW 0.157  OA 0.224
za_karoo_shrub mIoU vs DW 0.158  OA 0.613
za_capetown  mIoU vs DW 0.482  OA 0.887
australia_outback mIoU vs DW 0.204  OA 0.718
sahel_niger  mIoU vs DW 0.039  OA 0.180
patagonia_steppe mIoU vs DW 0.090  OA 0.495
nz_pasture   mIoU vs DW 0.244  OA 0.573
scottish_highlands mIoU vs DW 0.216  OA 0.565
iceland_tundra mIoU vs DW 0.099  OA 0.305
tibetan_plateau mIoU vs DW 0.154  OA 0.767
thai_paddy   mIoU vs DW 0.281  OA 0.760
madagascar_plateau mIoU vs DW 0.